# Data Engineering Interview Prep: Easy SQL (Complete 1-21)
## Topic: Filtering, Joins, Aggregations, and Date Math

---

### **Easy Lesson 1: Twitter - "Histogram of Tweets"**

#### **The Problem**
Write a query to obtain a histogram of tweets posted per user in 2022. Output the tweet bucket and the number of users in that bucket.

#### **The Logic**
Use a two-step aggregation: first, count the tweets per user in 2022. Then, group by those counts to find how many users fall into each bucket.

#### **The Solution (PostgreSQL)**
```sql
WITH tweet_counts AS (
  SELECT user_id, COUNT(tweet_id) AS tweet_bucket
  FROM tweets
  WHERE tweet_date >= '2022-01-01' AND tweet_date < '2023-01-01'
  GROUP BY user_id
)
SELECT tweet_bucket, COUNT(user_id) AS users_num
FROM tweet_counts
GROUP BY tweet_bucket
ORDER BY tweet_bucket;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For column filtering, grouping, and aggregations).
```python
import pyspark.sql.functions as F

tweet_counts = tweets.filter(
    (F.col("tweet_date") >= "2022-01-01") & (F.col("tweet_date") < "2023-01-01")
).groupBy("user_id").agg(
    F.count("tweet_id").alias("tweet_bucket")
)

result = tweet_counts.groupBy("tweet_bucket").agg(
    F.count("user_id").alias("users_num")
).orderBy("tweet_bucket")
```

#### **Senior Data Engineer Perspective**
Ensure massive tables are partitioned by `tweet_date` for partition pruning. Be aware of data skew during the first `GROUP BY` if "Power Users" have significantly more tweets than standard users.

---

### **Easy Lesson 2: LinkedIn - "Data Science Skills"**

#### **The Problem**
List candidate IDs who possess Python, Tableau, and PostgreSQL. Order ascending.

#### **The Logic**
Filter for the required skills, group by the candidate, and use `HAVING` to ensure they possess exactly 3 distinct required skills.

#### **The Solution (PostgreSQL)**
```sql
SELECT candidate_id
FROM candidates
WHERE skill IN ('Python', 'Tableau', 'PostgreSQL')
GROUP BY candidate_id
HAVING COUNT(skill) = 3
ORDER BY candidate_id ASC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For `.isin()`, `countDistinct`, and filtering).
```python
import pyspark.sql.functions as F

result = candidates.filter(
    F.col("skill").isin("Python", "Tableau", "PostgreSQL")
).groupBy("candidate_id").agg(
    F.countDistinct("skill").alias("skill_count")
).filter(F.col("skill_count") == 3).select("candidate_id").orderBy("candidate_id")
```

#### **Senior Data Engineer Perspective**
The `IN` clause enables predicate pushdown, minimizing data sent to the shuffle phase. Using `COUNT(DISTINCT skill)` is safer to protect against duplicate skill entries in messy source data.

---

### **Easy Lesson 3: Facebook - "Page With No Likes"**

#### **The Problem**
Return the IDs of Facebook pages that have zero likes.

#### **The Logic**
Perform a `LEFT JOIN` from pages to likes and filter for where the likes side `IS NULL`.

#### **The Solution (PostgreSQL)**
```sql
SELECT p.page_id
FROM pages p
LEFT JOIN page_likes pl ON p.page_id = pl.page_id
WHERE pl.page_id IS NULL
ORDER BY p.page_id ASC;
```

#### **The Solution (PySpark)**
**Modules Used:** No specific external modules needed, relies on native DataFrame `.join()` methodology.
```python
# A left_anti join is PySpark's highly optimized native method for finding missing records
result = pages.join(
    page_likes, 
    pages.page_id == page_likes.page_id, 
    "left_anti"
).select(pages.page_id).orderBy(pages.page_id)
```

#### **Senior Data Engineer Perspective**
Prefer `LEFT JOIN ... IS NULL` over `NOT IN`, as `NOT IN` fails if the subquery returns any `NULL` values. In PySpark, this is executed efficiently as a `left_anti` join.

---

### **Easy Lesson 4: Tesla - "Unfinished Parts"**

#### **The Problem**
Find parts that have begun assembly but lack a finish date.

#### **The Logic**
Filter the table for rows where the `finish_date` is missing.

#### **The Solution (PostgreSQL)**
```sql
SELECT part, assembly_step
FROM parts_assembly
WHERE finish_date IS NULL;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For column null checking).
```python
import pyspark.sql.functions as F

result = parts_assembly.filter(
    F.col("finish_date").isNull()
).select("part", "assembly_step")
```

#### **Senior Data Engineer Perspective**
Columnar formats like Parquet track null counts in their metadata, making `IS NULL` checks fast. If queried frequently, partitioning by an `is_finished` boolean flag is better than scanning for nulls.

---

### **Easy Lesson 5: NY Times - "Laptop vs. Mobile Viewership"**

#### **The Problem**
Calculate total viewership for laptops vs. mobile (tablet + phone).

#### **The Logic**
Use conditional aggregation (`SUM(CASE WHEN...)`) to route the counts in a single pass.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  SUM(CASE WHEN device_type = 'laptop' THEN 1 ELSE 0 END) AS laptop_views,
  SUM(CASE WHEN device_type IN ('tablet', 'phone') THEN 1 ELSE 0 END) AS mobile_views
FROM viewership;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For conditional `when/otherwise` inside an aggregation).
```python
import pyspark.sql.functions as F

result = viewership.select(
    F.sum(F.when(F.col("device_type") == "laptop", 1).otherwise(0)).alias("laptop_views"),
    F.sum(F.when(F.col("device_type").isin("tablet", "phone"), 1).otherwise(0)).alias("mobile_views")
)
```

#### **Senior Data Engineer Perspective**
Conditional aggregation scans the table exactly once. In Spark, `F.sum(F.when(...))` is more performant than using `groupBy().pivot()` because it avoids dynamic pivoting overhead.

---

### **Easy Lesson 6: Facebook - "Average Post Hiatus (Part 1)"**

#### **The Problem**
Find the days between a user's first and last post in 2021 for users with at least 2 posts.

#### **The Logic**
Group by user, apply a `HAVING` clause for the count, and subtract the min date from the max date.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  user_id, 
  EXTRACT(DAY FROM MAX(post_date) - MIN(post_date)) AS days_between
FROM posts
WHERE post_date >= '2021-01-01' AND post_date < '2022-01-01'
GROUP BY user_id
HAVING COUNT(post_id) >= 2;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For date math using `datediff`).
```python
import pyspark.sql.functions as F

result = posts.filter(
    (F.col("post_date") >= "2021-01-01") & (F.col("post_date") < "2022-01-01")
).groupBy("user_id").agg(
    F.count("post_id").alias("post_count"),
    F.datediff(F.max("post_date"), F.min("post_date")).alias("days_between")
).filter(F.col("post_count") >= 2).select("user_id", "days_between")
```

#### **Senior Data Engineer Perspective**
Applying functions to columns in the `WHERE` clause (like `EXTRACT(YEAR FROM date)`) ruins index usage. Always use raw date bounds for sargable queries.

---

### **Easy Lesson 7: Microsoft - "Teams Power Users"**

#### **The Problem**
Identify the top 2 users who sent the most messages in August 2022.

#### **The Logic**
Group by sender, count messages, order descending, and limit to 2.

#### **The Solution (PostgreSQL)**
```sql
SELECT sender_id, COUNT(message_id) AS message_count
FROM messages
WHERE sent_date >= '2022-08-01' AND sent_date < '2022-09-01'
GROUP BY sender_id
ORDER BY message_count DESC
LIMIT 2;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For aggregation and descending order).
```python
import pyspark.sql.functions as F

result = messages.filter(
    (F.col("sent_date") >= "2022-08-01") & (F.col("sent_date") < "2022-09-01")
).groupBy("sender_id").agg(
    F.count("message_id").alias("message_count")
).orderBy(F.col("message_count").desc()).limit(2)
```

#### **Senior Data Engineer Perspective**
Global `ORDER BY` in distributed systems pushes all data to a single node. Query engines usually optimize small limits using local "Top-K" algorithms before merging.

---

### **Easy Lesson 8: LinkedIn - "Duplicate Job Listings"**

#### **The Problem**
Count companies that posted multiple jobs with the exact same title and description.

#### **The Logic**
Group by company, title, and description, filter for counts > 1, then count the distinct companies from that result.

#### **The Solution (PostgreSQL)**
```sql
WITH duplicates AS (
  SELECT company_id
  FROM job_listings
  GROUP BY company_id, title, description
  HAVING COUNT(job_id) > 1
)
SELECT COUNT(DISTINCT company_id) AS duplicate_companies
FROM duplicates;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

duplicates = job_listings.groupBy("company_id", "title", "description").agg(
    F.count("job_id").alias("job_count")
).filter(F.col("job_count") > 1)

result = duplicates.select("company_id").distinct().select(
    F.count("company_id").alias("duplicate_companies")
)
```

#### **Senior Data Engineer Perspective**
Grouping by massive text blocks destroys memory. Generate an MD5/SHA-256 hash of the text during ingestion and group by the hash instead.

---

### **Easy Lesson 9: Robinhood - "Cities With Completed Trades"**

#### **The Problem**
Retrieve the top three cities with the most 'Completed' trades.

#### **The Logic**
Join trades to users, filter for 'Completed', group by city, sort, and limit.

#### **The Solution (PostgreSQL)**
```sql
SELECT u.city, COUNT(t.order_id) AS total_orders
FROM trades t
JOIN users u ON t.user_id = u.user_id
WHERE t.status = 'Completed'
GROUP BY u.city
ORDER BY total_orders DESC
LIMIT 3;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (Specifically utilizing `F.broadcast` to optimize the join).
```python
import pyspark.sql.functions as F

result = trades.filter(F.col("status") == "Completed").join(
    F.broadcast(users), # Forces a Broadcast Hash Join for the smaller table
    "user_id", 
    "inner"
).groupBy("city").agg(
    F.count("order_id").alias("total_orders")
).orderBy(F.col("total_orders").desc()).limit(3)
```

#### **Senior Data Engineer Perspective**
This is a classic Fact-to-Dimension join. Ensure the smaller `users` table is broadcasted across worker nodes (Broadcast Hash Join) to prevent shuffling the massive `trades` table.

---

### **Easy Lesson 10: Amazon - "Average Review Ratings"**

#### **The Problem**
Get the average star rating per product, grouped by month.

#### **The Logic**
Extract the month, group by month and product, and round the average.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  EXTRACT(MONTH FROM submit_date) AS mth,
  product_id,
  ROUND(AVG(stars), 2) AS avg_stars
FROM reviews
GROUP BY EXTRACT(MONTH FROM submit_date), product_id
ORDER BY mth, product_id;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For date extraction `month()` and mathematical rounding).
```python
import pyspark.sql.functions as F

result = reviews.withColumn("mth", F.month("submit_date")).groupBy("mth", "product_id").agg(
    F.round(F.avg("stars"), 2).alias("avg_stars")
).orderBy("mth", "product_id")
```

#### **Senior Data Engineer Perspective**
Do not run heavy aggregations directly against source tables for BI dashboards. Schedule an Airflow/Spark job to calculate these and write to a Materialized View.

---

### **Easy Lesson 11: FAANG - "Well Paid Employees"**

#### **The Problem**
Identify employees who earn more than their direct managers.

#### **The Logic**
Perform a self-join comparing the employee's manager_id to the manager's employee_id.

#### **The Solution (PostgreSQL)**
```sql
SELECT e.name AS employee_name
FROM employee e
JOIN employee m ON e.manager_id = m.employee_id
WHERE e.salary > m.salary;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For DataFrame aliasing to perform a clean self-join).
```python
import pyspark.sql.functions as F

e = employee.alias("e")
m = employee.alias("m")

result = e.join(
    m, 
    F.col("e.manager_id") == F.col("m.employee_id"), 
    "inner"
).filter(
    F.col("e.salary") > F.col("m.salary")
).select(F.col("e.name").alias("employee_name"))
```

#### **Senior Data Engineer Perspective**
Self-joins on large hierarchy tables trigger massive network shuffles. Flatten hierarchies into arrays or use Spark GraphFrames for large-scale organizational data.

---

### **Easy Lesson 12: PayPal - "Final Account Balance"**

#### **The Problem**
Calculate final balance from a ledger of deposits and withdrawals.

#### **The Logic**
Use conditional math inside a `SUM()` to add deposits and subtract withdrawals.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  account_id,
  SUM(CASE 
      WHEN transaction_type = 'Deposit' THEN amount 
      WHEN transaction_type = 'Withdrawal' THEN -amount 
      ELSE 0 END) AS final_balance
FROM transactions
GROUP BY account_id;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For mapping the `CASE WHEN` logic to `when().otherwise()`).
```python
import pyspark.sql.functions as F

result = transactions.groupBy("account_id").agg(
    F.sum(
        F.when(F.col("transaction_type") == "Deposit", F.col("amount"))
         .when(F.col("transaction_type") == "Withdrawal", -F.col("amount"))
         .otherwise(0)
    ).alias("final_balance")
)
```

#### **Senior Data Engineer Perspective**
Financial pipelines must be strictly idempotent. Ensure the architecture relies on processed-event tracking to prevent double-counting if a worker node crashes mid-execution.

---

### **Easy Lesson 13: Facebook - "App Click-through Rate (CTR)"**

#### **The Problem**
Calculate CTR (Clicks / Impressions) for apps in 2022.

#### **The Logic**
Use conditional sums to isolate clicks and impressions, then divide.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  app_id,
  ROUND(100.0 * SUM(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) / 
    NULLIF(SUM(CASE WHEN event_type = 'impression' THEN 1 ELSE 0 END), 0), 2) AS ctr
FROM events
WHERE timestamp >= '2022-01-01' AND timestamp < '2023-01-01'
GROUP BY app_id;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

result = events.filter(
    (F.col("timestamp") >= "2022-01-01") & (F.col("timestamp") < "2023-01-01")
).groupBy("app_id").agg(
    F.round(
        100.0 * F.sum(F.when(F.col("event_type") == "click", 1).otherwise(0)) /
        F.sum(F.when(F.col("event_type") == "impression", 1).otherwise(0)), 
        2
    ).alias("ctr")
)
```

#### **Senior Data Engineer Perspective**
`NULLIF(val, 0)` is mandatory. Without it, anomalous data where impressions equal zero will cause a fatal Division by Zero error.

---

### **Easy Lesson 14: TikTok - "Second Day Confirmation"**

#### **The Problem**
Find users who confirmed sign-up exactly one day after initiating it.

#### **The Logic**
Join the tables and filter where action date equals signup date plus a 1-day interval.

#### **The Solution (PostgreSQL)**
```sql
SELECT e.user_id
FROM emails e
JOIN texts t ON e.email_id = t.email_id
WHERE t.signup_action = 'Confirmed'
  AND t.action_date = e.signup_date + INTERVAL '1 day';
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For adding intervals using `date_add()`).
```python
import pyspark.sql.functions as F

result = emails.join(
    texts, 
    "email_id", 
    "inner"
).filter(
    (F.col("signup_action") == "Confirmed") & 
    (F.col("action_date") == F.date_add(F.col("signup_date"), 1))
).select("user_id")
```

#### **Senior Data Engineer Perspective**
Standardize all ingestion to UTC. When building date logic, differentiate between strict 24-hour windows and calendar midnight crossings.

---

### **Easy Lesson 15: IBM - "IBM db2 Product Analytics"**

#### **The Problem**
Histogram of unique queries per employee in Q3, including 0-query employees.

#### **The Logic**
`LEFT JOIN` employees to queries, ensuring the date filter is inside the `ON` clause, not `WHERE`.

#### **The Solution (PostgreSQL)**
```sql
WITH query_counts AS (
  SELECT e.emp_id, COUNT(q.query_id) AS query_count
  FROM employees e
  LEFT JOIN queries q 
    ON e.emp_id = q.emp_id 
    AND q.query_date >= '2023-07-01' AND q.query_date < '2023-10-01'
  GROUP BY e.emp_id
)
SELECT query_count AS unique_queries, COUNT(emp_id) AS employee_count
FROM query_counts
GROUP BY query_count
ORDER BY unique_queries;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

# Filter the right table BEFORE the left join to mimic the SQL 'ON' clause behavior
q_filtered = queries.filter(
    (F.col("query_date") >= "2023-07-01") & (F.col("query_date") < "2023-10-01")
)

query_counts = employees.join(
    q_filtered, 
    "emp_id", 
    "left"
).groupBy("emp_id").agg(
    F.count("query_id").alias("query_count")
)

result = query_counts.groupBy("query_count").agg(
    F.count("emp_id").alias("employee_count")
).withColumnRenamed("query_count", "unique_queries").orderBy("unique_queries")
```

#### **Senior Data Engineer Perspective**
The date filter must live in the `ON` clause. If placed in the `WHERE` clause, null `query_date` rows are filtered out, turning the `LEFT JOIN` into an `INNER JOIN`.

---

### **Easy Lesson 16: JPMorgan - "Cards Issued Difference"**

#### **The Problem**
Find the difference between max and min issued amounts for each card.

#### **The Logic**
Group by card name and subtract `MIN()` from `MAX()`.

#### **The Solution (PostgreSQL)**
```sql
SELECT card_name, MAX(issued_amount) - MIN(issued_amount) AS difference
FROM monthly_cards_issued
GROUP BY card_name
ORDER BY difference DESC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

result = monthly_cards_issued.groupBy("card_name").agg(
    (F.max("issued_amount") - F.min("issued_amount")).alias("difference")
).orderBy(F.col("difference").desc())
```

#### **Senior Data Engineer Perspective**
Query engines using Parquet or ORC can execute MIN/MAX aggregations by simply reading file footer metadata, bypassing row-level scanning entirely.

---

### **Easy Lesson 17: Alibaba - "Compressed Mean"**

#### **The Problem**
Find mean items per order given a pre-aggregated table of counts.

#### **The Logic**
Multiply counts by occurrences, sum them, cast to decimal, and divide by total occurrences.

#### **The Solution (PostgreSQL)**
```sql
SELECT ROUND(
    SUM(item_count::DECIMAL * order_occurrences) / SUM(order_occurrences)
  , 1) AS mean
FROM items_per_order;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

result = items_per_order.select(
    F.round(
        F.sum(F.col("item_count").cast("decimal") * F.col("order_occurrences")) / 
        F.sum("order_occurrences"), 
        1
    ).alias("mean")
)
```

#### **Senior Data Engineer Perspective**
Prevent integer truncation bugs by explicitly casting the numerator to `DECIMAL`. Storing data in this "compressed" format is a standard Data Warehousing pre-aggregation technique.

---

### **Easy Lesson 18: CVS Health - "Pharmacy Analytics (Part 1)"**

#### **The Problem**
Find the top 3 most profitable drugs.

#### **The Logic**
Subtract cogs from sales directly, sort descending, and limit.

#### **The Solution (PostgreSQL)**
```sql
SELECT drug, (total_sales - cogs) AS total_profit
FROM pharmacy_sales
ORDER BY total_profit DESC
LIMIT 3;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

result = pharmacy_sales.withColumn(
    "total_profit", 
    F.col("total_sales") - F.col("cogs")
).select("drug", "total_profit").orderBy(F.col("total_profit").desc()).limit(3)
```

#### **Senior Data Engineer Perspective**
Row-level math is extremely fast in Spark. Project Tungsten optimizes this using SIMD (Single Instruction, Multiple Data) execution directly in CPU registers.

---

### **Easy Lesson 19: CVS Health - "Pharmacy Analytics (Part 2)"**

#### **The Problem**
Identify manufacturers operating at a loss, showing drug count and total loss.

#### **The Logic**
Filter for losses first, then group by manufacturer and aggregate.

#### **The Solution (PostgreSQL)**
```sql
SELECT manufacturer, COUNT(drug) AS drug_count, SUM(cogs - total_sales) AS total_loss
FROM pharmacy_sales
WHERE cogs > total_sales
GROUP BY manufacturer
ORDER BY total_loss DESC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

result = pharmacy_sales.filter(
    F.col("cogs") > F.col("total_sales")
).groupBy("manufacturer").agg(
    F.count("drug").alias("drug_count"),
    F.sum(F.col("cogs") - F.col("total_sales")).alias("total_loss")
).orderBy(F.col("total_loss").desc())
```

#### **Senior Data Engineer Perspective**
Predicate pushdown priority is key here. Filtering out profitable drugs *before* the `GROUP BY` drastically reduces memory overhead during the shuffle phase.

---

### **Easy Lesson 20: CVS Health - "Pharmacy Analytics (Part 3)"**

#### **The Problem**
Format total sales into a string like "$36 million".

#### **The Logic**
Sum the sales, divide by 1M, round, and wrap in a `CONCAT()` function.

#### **The Solution (PostgreSQL)**
```sql
SELECT manufacturer, CONCAT('$', ROUND(SUM(total_sales) / 1000000), ' million') AS sale_format
FROM pharmacy_sales
GROUP BY manufacturer
ORDER BY SUM(total_sales) DESC, manufacturer ASC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F` (For string concatenation via `concat` and formatting logic).
```python
import pyspark.sql.functions as F

result = pharmacy_sales.groupBy("manufacturer").agg(
    F.sum("total_sales").alias("sum_sales")
).orderBy(
    F.col("sum_sales").desc(), 
    F.col("manufacturer").asc()
).withColumn(
    "sale_format", 
    F.concat(F.lit("$"), F.round(F.col("sum_sales") / 1000000), F.lit(" million"))
).select("manufacturer", "sale_format")
```

#### **Senior Data Engineer Perspective**
Separate the data and presentation layers. Provide pure numeric outputs to the BI tool and handle string formatting there to allow for downstream math and sorting.

---

### **Easy Lesson 21: UnitedHealth - "Patient Support Analysis (Part 1)"**

#### **The Problem**
Count total policyholders who called 3 or more times.

#### **The Logic**
Use a CTE to find policyholders with >= 3 calls, then do a global count of those users.

#### **The Solution (PostgreSQL)**
```sql
WITH heavy_callers AS (
  SELECT policy_holder_id
  FROM callers
  GROUP BY policy_holder_id
  HAVING COUNT(case_id) >= 3
)
SELECT COUNT(policy_holder_id) AS member_count
FROM heavy_callers;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

heavy_callers = callers.groupBy("policy_holder_id").agg(
    F.count("case_id").alias("case_count")
).filter(F.col("case_count") >= 3)

result = heavy_callers.select(F.count("policy_holder_id").alias("member_count"))
```

#### **Senior Data Engineer Perspective**
A `COUNT()` without a `GROUP BY` requires all data to funnel into a single executor node. If the intermediate CTE result is massive, this causes Out-Of-Memory exceptions.

# Data Engineering Interview Prep: Medium SQL (Complete 1-19)
## Topic: Window Functions, Relational Division, and Time-Series Analytics

---

### **Medium Lesson 1: Uber - "User's Third Transaction"**

#### **The Problem**
Write a query to obtain the third transaction of every user. Output the `user_id`, `spend`, and `transaction_date`.

#### **The Logic (Window Functions)**
Rank each user's transactions chronologically using `ROW_NUMBER()` partitioned by the user.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_transactions AS (
  SELECT 
    user_id, 
    spend, 
    transaction_date,
    ROW_NUMBER() OVER (
      PARTITION BY user_id 
      ORDER BY transaction_date
    ) AS transaction_rank
  FROM transactions
)
SELECT user_id, spend, transaction_date
FROM ranked_transactions
WHERE transaction_rank = 3;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("user_id").orderBy("transaction_date")

result = transactions.withColumn(
    "transaction_rank", 
    F.row_number().over(window_spec)
).filter(F.col("transaction_rank") == 3).drop("transaction_rank")
```

#### **Senior Data Engineer Perspective**
In PySpark, `Window.partitionBy("user_id")` requires a massive data shuffle. All transactions for a specific user must be moved across the network to reside on the exact same worker node. Beware of data skew (e.g., corporate accounts with 100x more transactions than normal users) which can cause Out-Of-Memory (OOM) errors. 

---

### **Medium Lesson 2: FAANG - "Second Highest Salary"**

#### **The Problem**
Find the second-highest salary from an `employee` table. Return `NULL` if none exists.

#### **The Solution (PostgreSQL)**
```sql
SELECT MAX(salary) AS second_highest_salary
FROM employee
WHERE salary < (
  SELECT MAX(salary) 
  FROM employee
);
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# In PySpark, using Dense Rank is often safer and more native than simulating scalar subqueries
window_spec = Window.orderBy(F.col("salary").desc())

result = employee.withColumn(
    "rank", 
    F.dense_rank().over(window_spec)
).filter(F.col("rank") == 2).select(
    F.max("salary").alias("second_highest_salary")
)
```

#### **Senior Data Engineer Perspective**
Avoid `ORDER BY salary DESC LIMIT 1 OFFSET 1`. Global sorting in distributed systems pushes all data to a single node. The `MAX()` approach acts as a Map-Reduce operation, where each node independently calculates its local maximum, which is vastly more scalable.

---

### **Medium Lesson 3: Snapchat - "Sending vs Opening Snaps"**

#### **The Problem**
Calculate the percentage of time spent sending versus opening snaps for each age group. Round to 2 decimal places.

#### **The Solution (PostgreSQL)**
```sql
WITH snap_stats AS (
  SELECT 
    ab.age_bucket,
    SUM(CASE WHEN a.activity_type = 'send' THEN a.time_spent ELSE 0 END) AS send_time,
    SUM(CASE WHEN a.activity_type = 'open' THEN a.time_spent ELSE 0 END) AS open_time,
    SUM(CASE WHEN a.activity_type IN ('send', 'open') THEN a.time_spent ELSE 0 END) AS total_time
  FROM activities a
  JOIN age_breakdown ab ON a.user_id = ab.user_id
  GROUP BY ab.age_bucket
)
SELECT 
  age_bucket,
  ROUND(100.0 * send_time / total_time, 2) AS send_perc,
  ROUND(100.0 * open_time / total_time, 2) AS open_perc
FROM snap_stats;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

snap_stats = activities.join(
    F.broadcast(age_breakdown), # Broadcast the small dimension table
    "user_id",
    "inner"
).groupBy("age_bucket").agg(
    F.sum(F.when(F.col("activity_type") == "send", F.col("time_spent")).otherwise(0)).alias("send_time"),
    F.sum(F.when(F.col("activity_type") == "open", F.col("time_spent")).otherwise(0)).alias("open_time"),
    F.sum(F.when(F.col("activity_type").isin("send", "open"), F.col("time_spent")).otherwise(0)).alias("total_time")
)

result = snap_stats.select(
    "age_bucket",
    F.round((100.0 * F.col("send_time")) / F.col("total_time"), 2).alias("send_perc"),
    F.round((100.0 * F.col("open_time")) / F.col("total_time"), 2).alias("open_perc")
)
```

#### **Senior Data Engineer Perspective**
`activities` is a massive fact table, while `age_breakdown` is a small dimension table. Ensure the query optimizer executes a Broadcast Hash Join to prevent shuffling the massive activities table over the network.

---

### **Medium Lesson 4: Twitter - "Tweets' Rolling Averages"**

#### **The Problem**
Calculate the 3-day rolling average of tweets for each user over a specified time period. Round to 2 decimal places.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  user_id, 
  tweet_date,
  ROUND(
    AVG(tweet_count) OVER (
      PARTITION BY user_id 
      ORDER BY tweet_date 
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2
  ) AS rolling_avg_3d
FROM tweets;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Define the sliding window frame
window_spec = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(-2, Window.currentRow)

result = tweets.select(
    "user_id",
    "tweet_date",
    F.round(F.avg("tweet_count").over(window_spec), 2).alias("rolling_avg_3d")
)
```

#### **Senior Data Engineer Perspective**
The `ROWS BETWEEN` frame logic is identical to what is used in real-time frameworks like Spark Structured Streaming. When calculating rolling metrics on live streams, you must also define Watermarking to handle late-arriving data effectively before the window calculation finalizes.

---

### **Medium Lesson 5: Amazon - "Highest-Grossing Items"**

#### **The Problem**
Identify the top two highest-grossing products within each category in 2022. 

#### **The Solution (PostgreSQL)**
```sql
WITH category_totals AS (
  SELECT 
    category, 
    product, 
    SUM(spend) AS total_spend
  FROM product_spend
  WHERE EXTRACT(YEAR FROM transaction_date) = 2022
  GROUP BY category, product
),
ranked_spend AS (
  SELECT 
    category, 
    product, 
    total_spend,
    RANK() OVER (PARTITION BY category ORDER BY total_spend DESC) AS rank_num
  FROM category_totals
)
SELECT category, product, total_spend
FROM ranked_spend
WHERE rank_num <= 2;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

category_totals = product_spend.filter(
    F.year("transaction_date") == 2022
).groupBy("category", "product").agg(
    F.sum("spend").alias("total_spend")
)

window_spec = Window.partitionBy("category").orderBy(F.col("total_spend").desc())

result = category_totals.withColumn(
    "rank_num", 
    F.rank().over(window_spec)
).filter(F.col("rank_num") <= 2).drop("rank_num")
```

#### **Senior Data Engineer Perspective**
In PySpark, calculating ranks requires partitioning the data by `category`. If one category has billions of rows while another has a few thousand, you will hit severe **Data Skew**. In a Senior DE role, handle this by using a two-pass aggregation strategy to avoid Out-Of-Memory errors on the skewed partition.

---

### **Medium Lesson 6: FAANG - "Top Three Salaries"**

#### **The Problem**
Find the top 3 highest-paid employees in each department. 

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_salaries AS (
  SELECT 
    d.department_name, 
    e.name, 
    e.salary,
    DENSE_RANK() OVER (PARTITION BY d.department_id ORDER BY e.salary DESC) AS salary_rank
  FROM employee e
  JOIN department d ON e.department_id = d.department_id
)
SELECT department_name, name, salary
FROM ranked_salaries
WHERE salary_rank <= 3;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("department_id").orderBy(F.col("salary").desc())

# Window FIRST, filter, then Join (Highly optimized for distributed systems)
top_employees = employee.withColumn(
    "salary_rank", 
    F.dense_rank().over(window_spec)
).filter(F.col("salary_rank") <= 3)

result = top_employees.join(
    F.broadcast(department), 
    "department_id", 
    "inner"
).select("department_name", "name", "salary")
```

#### **Senior Data Engineer Perspective**
In a massive data lake ecosystem, the `employee` table could be huge. It is vastly faster to compute the `DENSE_RANK()` entirely within the massive `employee` table first, filter it down, and *then* join the resulting tiny dataset to the `department` table.

---

### **Medium Lesson 7: TikTok - "Signup Activation Rate"**

#### **The Problem**
Calculate the activation rate of TikTok users, rounded to 2 decimal places. 

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  ROUND(
    COUNT(t.email_id)::DECIMAL / COUNT(DISTINCT e.user_id)
  , 2) AS confirm_rate
FROM emails e
LEFT JOIN texts t 
  ON e.email_id = t.email_id 
  AND t.signup_action = 'Confirmed';
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

t_confirmed = texts.filter(F.col("signup_action") == "Confirmed")

joined_df = emails.join(
    t_confirmed, 
    "email_id", 
    "left"
)

result = joined_df.select(
    F.round(
        F.count("email_id").cast("decimal") / F.countDistinct("user_id"), 
        2
    ).alias("confirm_rate")
)
```

#### **Senior Data Engineer Perspective**
A Senior DE must ask: *"Can a user have multiple confirmed text events?"* If the source system retries SMS deliveries, a basic `COUNT()` would artificially inflate the numerator. Using `COUNT(DISTINCT t.email_id)` is the safer, defensive programming approach.

---

### **Medium Lesson 8: Spotify - "Spotify Streaming History"**

#### **The Problem**
Output the cumulative count of song plays up to August 4th, 2022 by combining historical and weekly logs.

#### **The Solution (PostgreSQL)**
```sql
WITH combined_data AS (
  SELECT user_id, song_id, song_plays FROM songs_history
  UNION ALL
  SELECT user_id, song_id, COUNT(song_id) AS song_plays
  FROM songs_weekly
  WHERE listen_time <= '2022-08-04 23:59:59'
  GROUP BY user_id, song_id
)
SELECT user_id, song_id, SUM(song_plays) AS total_plays
FROM combined_data
GROUP BY user_id, song_id
ORDER BY total_plays DESC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

weekly_agg = songs_weekly.filter(
    F.col("listen_time") <= "2022-08-04 23:59:59"
).groupBy("user_id", "song_id").agg(
    F.count("song_id").alias("song_plays")
)

# Combine the batch table with the newly aggregated streaming table
combined_data = songs_history.select("user_id", "song_id", "song_plays").unionAll(
    weekly_agg.select("user_id", "song_id", "song_plays")
)

result = combined_data.groupBy("user_id", "song_id").agg(
    F.sum("song_plays").alias("total_plays")
).orderBy(F.col("total_plays").desc())
```

#### **Senior Data Engineer Perspective**
This perfectly mimics a real-world **Lambda Architecture**. You have a "Batch Layer" (`songs_history`) and a "Speed Layer" (`songs_weekly`). A Senior DE seamlessly stitches these two data stores together to give downstream analysts a unified view without recalculating history.

---

### **Medium Lesson 9: Microsoft - "Supercloud Customer"**

#### **The Problem**
Find customers who bought at least one product from *every* product category.

#### **The Solution (PostgreSQL)**
```sql
SELECT c.customer_id
FROM customer_contracts c
JOIN products p ON c.product_id = p.product_id
GROUP BY c.customer_id
HAVING COUNT(DISTINCT p.product_category) = (
  SELECT COUNT(DISTINCT product_category) FROM products
);
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

# Extract the total category count as an integer first
total_categories = products.select(F.countDistinct("product_category")).collect()[0][0]

result = customer_contracts.join(
    products, 
    "product_id", 
    "inner"
).groupBy("customer_id").agg(
    F.countDistinct("product_category").alias("cat_count")
).filter(F.col("cat_count") == total_categories).select("customer_id")
```

#### **Senior Data Engineer Perspective**
Dynamic thresholds are critical. In Spark, executing the subquery via `collect()[0][0]` gets the exact integer. The optimizer evaluates that subquery once, broadcasts the integer to all workers, and executes the `GROUP BY` efficiently without hardcoding logic.

---

### **Medium Lesson 10: Google - "Odd and Even Measurements"**

#### **The Problem**
Calculate the sum of odd-numbered and even-numbered measurements chronologically for a particular day.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_measurements AS (
  SELECT 
    CAST(measurement_time AS DATE) AS measurement_day, 
    measurement_value, 
    ROW_NUMBER() OVER (PARTITION BY CAST(measurement_time AS DATE) ORDER BY measurement_time) AS measurement_num 
  FROM measurements
)
SELECT 
  measurement_day, 
  SUM(CASE WHEN measurement_num % 2 != 0 THEN measurement_value ELSE 0 END) AS odd_sum,
  SUM(CASE WHEN measurement_num % 2 = 0 THEN measurement_value ELSE 0 END) AS even_sum
FROM ranked_measurements
GROUP BY measurement_day
ORDER BY measurement_day;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

df = measurements.withColumn("measurement_day", F.to_date("measurement_time"))

window_spec = Window.partitionBy("measurement_day").orderBy("measurement_time")

ranked_measurements = df.withColumn("measurement_num", F.row_number().over(window_spec))

result = ranked_measurements.groupBy("measurement_day").agg(
    F.sum(F.when(F.col("measurement_num") % 2 != 0, F.col("measurement_value")).otherwise(0)).alias("odd_sum"),
    F.sum(F.when(F.col("measurement_num") % 2 == 0, F.col("measurement_value")).otherwise(0)).alias("even_sum")
).orderBy("measurement_day")
```

#### **Senior Data Engineer Perspective**
Partitioning by a daily cast can still result in massive partitions. A Senior DE would often implement **minute or hour bucketing** upstream to keep the partitions manageable before doing sequential analysis.

---

### **Medium Lesson 11: Zomato - "Swapped Food Delivery"**

#### **The Problem**
Swap the `order_id` of consecutive orders (swap order 1 with 2, 3 with 4, etc.). If the total number is odd, the last ID remains the same.

#### **The Solution (PostgreSQL)**
```sql
WITH max_id_table AS (
  SELECT MAX(order_id) AS max_id FROM orders
)
SELECT 
  CASE 
    WHEN order_id % 2 != 0 AND order_id != (SELECT max_id FROM max_id_table) THEN order_id + 1
    WHEN order_id % 2 = 0 THEN order_id - 1
    ELSE order_id 
  END AS corrected_order_id,
  item_name
FROM orders
ORDER BY corrected_order_id ASC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

max_id = orders.select(F.max("order_id")).collect()[0][0]

result = orders.withColumn(
    "corrected_order_id",
    F.when((F.col("order_id") % 2 != 0) & (F.col("order_id") != max_id), F.col("order_id") + 1)
     .when(F.col("order_id") % 2 == 0, F.col("order_id") - 1)
     .otherwise(F.col("order_id"))
).select("corrected_order_id", "item_name").orderBy("corrected_order_id")
```

#### **Senior Data Engineer Perspective**
Generating sequential `order_id`s without gaps is notoriously difficult in distributed systems. If order IDs have gaps (e.g., 1, 2, 5, 6), this math-based approach completely breaks. A robust solution uses `LEAD()` and `LAG()` over chronological timestamps.

---

### **Medium Lesson 12: Bloomberg - "FAANG Stock Min-Max"**

#### **The Problem**
Find the highest and lowest closing prices for each FAANG stock, along with the corresponding dates.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_prices AS (
  SELECT ticker, date, close,
    ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY close DESC, date DESC) AS max_rank,
    ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY close ASC, date DESC) AS min_rank
  FROM stock_prices
)
SELECT max_prices.ticker, max_prices.close AS highest_price, max_prices.date AS highest_price_date,
       min_prices.close AS lowest_price, min_prices.date AS lowest_price_date
FROM (SELECT * FROM ranked_prices WHERE max_rank = 1) max_prices
JOIN (SELECT * FROM ranked_prices WHERE min_rank = 1) min_prices
  ON max_prices.ticker = min_prices.ticker;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

max_window = Window.partitionBy("ticker").orderBy(F.col("close").desc(), F.col("date").desc())
min_window = Window.partitionBy("ticker").orderBy(F.col("close").asc(), F.col("date").desc())

ranked = stock_prices.withColumn("max_rank", F.row_number().over(max_window)) \
                     .withColumn("min_rank", F.row_number().over(min_window))

max_df = ranked.filter(F.col("max_rank") == 1).select("ticker", F.col("close").alias("highest_price"), F.col("date").alias("highest_price_date"))
min_df = ranked.filter(F.col("min_rank") == 1).select("ticker", F.col("close").alias("lowest_price"), F.col("date").alias("lowest_price_date"))

result = max_df.join(min_df, "ticker", "inner")
```

#### **Senior Data Engineer Perspective**
Adding the date as a secondary sort is a crucial tie-breaker. If a stock hits its all-time high on two different days, `ROW_NUMBER()` needs a deterministic tie-breaker so the pipeline doesn't return randomly fluctuating dates on different runs.

---

### **Medium Lesson 13: Amazon - "Best-Selling Product"**

#### **The Problem**
Find the best-selling product (by highest quantity sold) for each month in 2022.

#### **The Solution (PostgreSQL)**
```sql
WITH monthly_sales AS (
  SELECT EXTRACT(MONTH FROM transaction_date) AS month, product_id, SUM(quantity) AS total_quantity
  FROM sales
  WHERE EXTRACT(YEAR FROM transaction_date) = 2022
  GROUP BY EXTRACT(MONTH FROM transaction_date), product_id
),
ranked_sales AS (
  SELECT month, product_id, total_quantity, RANK() OVER (PARTITION BY month ORDER BY total_quantity DESC) AS rnk
  FROM monthly_sales
)
SELECT month, product_id, total_quantity
FROM ranked_sales
WHERE rnk = 1;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

monthly_sales = sales.filter(F.year("transaction_date") == 2022).withColumn(
    "month", F.month("transaction_date")
).groupBy("month", "product_id").agg(
    F.sum("quantity").alias("total_quantity")
)

window_spec = Window.partitionBy("month").orderBy(F.col("total_quantity").desc())

result = monthly_sales.withColumn(
    "rnk", F.rank().over(window_spec)
).filter(F.col("rnk") == 1).drop("rnk")
```

#### **Senior Data Engineer Perspective**
`RANK()` allows ties. As a Senior DE, confirm stakeholder expectations: should the dashboard show both tying products, or should you implement a deterministic tie-breaker (like sorting by `product_id` alphabetically)?

---

### **Medium Lesson 14: Amazon - "User Shopping Sprees"**

#### **The Problem**
Identify users who made purchases on 3 or more consecutive days.

#### **The Solution (PostgreSQL)**
```sql
WITH unique_days AS (
  SELECT DISTINCT user_id, CAST(transaction_date AS DATE) AS buy_date FROM transactions
),
islands AS (
  SELECT user_id, buy_date, buy_date - DENSE_RANK() OVER (PARTITION BY user_id ORDER BY buy_date)::INT AS island_group
  FROM unique_days
)
SELECT DISTINCT user_id
FROM islands
GROUP BY user_id, island_group
HAVING COUNT(buy_date) >= 3;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

unique_days = transactions.withColumn("buy_date", F.to_date("transaction_date")) \
                          .select("user_id", "buy_date").distinct()

window_spec = Window.partitionBy("user_id").orderBy("buy_date")

# In PySpark, we use date_sub to subtract the dense_rank integer from the date
islands = unique_days.withColumn(
    "island_group", 
    F.expr("date_sub(buy_date, dense_rank() over (partition by user_id order by buy_date))")
)

result = islands.groupBy("user_id", "island_group").agg(
    F.count("buy_date").alias("streak")
).filter(F.col("streak") >= 3).select("user_id").distinct().orderBy("user_id")
```

#### **Senior Data Engineer Perspective**
Calculating consecutive days dynamically across history is expensive. Upstream Sessionization is better: when a user buys something, a microservice updates a `current_streak` counter in a fast NoSQL database (DynamoDB/Redis).

---

### **Medium Lesson 15: Walmart - "Histogram of Users and Purchases"**

#### **The Problem**
Find the most recent transaction date for each user and count how many items they purchased on that specific day.

#### **The Solution (PostgreSQL)**
```sql
WITH latest_transactions AS (
  SELECT user_id, transaction_date, product_id,
    RANK() OVER (PARTITION BY user_id ORDER BY transaction_date DESC) AS recent_rank
  FROM user_transactions
)
SELECT transaction_date, user_id, COUNT(product_id) AS purchase_count
FROM latest_transactions
WHERE recent_rank = 1
GROUP BY transaction_date, user_id;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("user_id").orderBy(F.col("transaction_date").desc())

latest_transactions = user_transactions.withColumn(
    "recent_rank", F.rank().over(window_spec)
).filter(F.col("recent_rank") == 1)

result = latest_transactions.groupBy("transaction_date", "user_id").agg(
    F.count("product_id").alias("purchase_count")
).orderBy("transaction_date")
```

#### **Senior Data Engineer Perspective**
Scanning an entire transaction log to find the latest date is highly inefficient. Senior engineers use SCD Type 2 tables with `is_current = TRUE` flags to query the "latest state" without window functions.

---

### **Medium Lesson 16: Alibaba - "Compressed Mode"**

#### **The Problem**
Find the mode (most frequent `item_count`) from a compressed table.

#### **The Solution (PostgreSQL)**
```sql
SELECT item_count
FROM items_per_order
WHERE order_occurrences = (SELECT MAX(order_occurrences) FROM items_per_order)
ORDER BY item_count ASC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

max_occ = items_per_order.select(F.max("order_occurrences")).collect()[0][0]

result = items_per_order.filter(
    F.col("order_occurrences") == max_occ
).select("item_count").orderBy("item_count")
```

#### **Senior Data Engineer Perspective**
This requires double scanning. In Parquet/ORC, that's fast because the MAX is in file metadata. If grouped by category, you would pivot to `MAX() OVER()` to avoid costly self-joins on a massive fact table.

---

### **Medium Lesson 17: JPMorgan - "Card Launch Success"**

#### **The Problem**
Identify the "launch month" and the number of cards issued in that month for each card name.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_launches AS (
  SELECT card_name, issued_amount,
    ROW_NUMBER() OVER (PARTITION BY card_name ORDER BY issue_year ASC, issue_month ASC) AS launch_rank
  FROM monthly_cards_issued
)
SELECT card_name, issued_amount
FROM ranked_launches
WHERE launch_rank = 1
ORDER BY issued_amount DESC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("card_name").orderBy("issue_year", "issue_month")

result = monthly_cards_issued.withColumn(
    "launch_rank", F.row_number().over(window_spec)
).filter(F.col("launch_rank") == 1).select(
    "card_name", "issued_amount"
).orderBy(F.col("issued_amount").desc())
```

#### **Senior Data Engineer Perspective**
Sorting by separate Year and Month integers is dangerous. A Senior DE would typically concatenate them into a proper `DATE` type upon ingestion so that time-series analysis is foolproof.

---

### **Medium Lesson 18: Verizon - "International Call Percentage"**

#### **The Problem**
Calculate the percentage of international calls.

#### **The Solution (PostgreSQL)**
```sql
SELECT ROUND(
    100.0 * SUM(CASE WHEN caller.country_id != receiver.country_id THEN 1 ELSE 0 END) / COUNT(*), 1
  ) AS international_calls_pct
FROM phone_calls calls
JOIN phone_info caller ON calls.caller_id = caller.caller_id
JOIN phone_info receiver ON calls.receiver_id = receiver.caller_id;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

calls_with_caller = phone_calls.join(
    phone_info.withColumnRenamed("country_id", "caller_country"),
    phone_calls.caller_id == phone_info.caller_id,
    "inner"
).drop(phone_info.caller_id)

calls_complete = calls_with_caller.join(
    phone_info.withColumnRenamed("country_id", "receiver_country"),
    calls_with_caller.receiver_id == phone_info.caller_id,
    "inner"
).drop(phone_info.caller_id)

result = calls_complete.select(
    F.round(
        100.0 * F.sum(F.when(F.col("caller_country") != F.col("receiver_country"), 1).otherwise(0)) / 
        F.count("*"), 
        1
    ).alias("international_calls_pct")
)
```

#### **Senior Data Engineer Perspective**
In a massive telecom environment, joining a multi-billion row CDR table to a subscriber dimension table twice triggers massive shuffles. Ensure the `phone_info` dimension is physically broadcasted to the workers.

---

### **Medium Lesson 19: UnitedHealth - "Patient Support Analysis (Part 2)"**

#### **The Problem**
Find the percentage of calls that cannot be categorized.

#### **The Solution (PostgreSQL)**
```sql
SELECT ROUND(
    100.0 * SUM(CASE WHEN call_category IS NULL OR call_category = 'n/a' THEN 1 ELSE 0 END) / COUNT(case_id), 1
  ) AS uncategorized_call_pct
FROM callers;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

result = callers.select(
    F.round(
        100.0 * F.sum(F.when(F.col("call_category").isNull() | (F.col("call_category") == "n/a"), 1).otherwise(0)) / 
        F.count("case_id"), 
        1
    ).alias("uncategorized_call_pct")
)
```

#### **Senior Data Engineer Perspective**
Handling `NULL` and string literals like `'n/a'` in analytical queries is a red flag. Implement a data quality framework to standardize missing categories into a single dimension key (e.g., `'Unknown'`) during the ETL phase.

# Data Engineering Interview Prep: Hard SQL (Complete 1-14)
## Topic: Retention, Hierarchies, Combinatorics, and Sequence Analytics

---

### **Hard Lesson 1: Facebook - "Active User Retention"**

#### **The Problem**
Calculate Monthly Active Users (MAUs) for July 2022. An active user performed actions in the current month (July) and the previous month (June).

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  EXTRACT(MONTH FROM curr_month.event_date) AS month, 
  COUNT(DISTINCT curr_month.user_id) AS monthly_active_users 
FROM user_actions curr_month 
WHERE EXTRACT(MONTH FROM curr_month.event_date) = 7 
  AND EXTRACT(YEAR FROM curr_month.event_date) = 2022 
  AND EXISTS (
    SELECT 1 
    FROM user_actions prev_month 
    WHERE curr_month.user_id = prev_month.user_id 
      AND EXTRACT(MONTH FROM prev_month.event_date) = 6 
      AND EXTRACT(YEAR FROM prev_month.event_date) = 2022
  )
GROUP BY EXTRACT(MONTH FROM curr_month.event_date);
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

# Filter for June and July separately
june_users = user_actions.filter(
    (F.month("event_date") == 6) & (F.year("event_date") == 2022)
).select("user_id").distinct()

july_users = user_actions.filter(
    (F.month("event_date") == 7) & (F.year("event_date") == 2022)
)

# A Left Semi Join acts identically to an EXISTS clause
result = july_users.join(
    june_users, 
    "user_id", 
    "left_semi"
).withColumn("month", F.month("event_date")).groupBy("month").agg(
    F.countDistinct("user_id").alias("monthly_active_users")
)
```

#### **Senior Data Engineer Perspective**
A self-join (or `EXISTS`) across two months of raw event logs is an anti-pattern in production. You would use an **Accumulating Snapshot table** tracking `first_active_date`, `last_active_date`, and an array of active months to turn a multi-terabyte shuffle into a simple filter query.

---

### **Hard Lesson 2: Wayfair - "Y-on-Y Growth Rate"**

#### **The Problem**
Calculate the year-on-year growth rate for the total spend of each product.

#### **The Solution (PostgreSQL)**
```sql
WITH yearly_spend AS (
  SELECT EXTRACT(YEAR FROM transaction_date) AS year, product_id, SUM(spend) AS curr_year_spend
  FROM user_transactions
  GROUP BY EXTRACT(YEAR FROM transaction_date), product_id
),
lagged_spend AS (
  SELECT year, product_id, curr_year_spend, 
    LAG(curr_year_spend) OVER (PARTITION BY product_id ORDER BY year ASC) AS prev_year_spend 
  FROM yearly_spend
)
SELECT year, product_id, curr_year_spend, prev_year_spend,
  ROUND(100.0 * (curr_year_spend - prev_year_spend) / prev_year_spend, 2) AS yoy_rate
FROM lagged_spend;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

yearly_spend = user_transactions.withColumn("year", F.year("transaction_date")) \
    .groupBy("year", "product_id").agg(F.sum("spend").alias("curr_year_spend"))

window_spec = Window.partitionBy("product_id").orderBy("year")

lagged_spend = yearly_spend.withColumn(
    "prev_year_spend", F.lag("curr_year_spend").over(window_spec)
)

result = lagged_spend.withColumn(
    "yoy_rate", 
    F.round(100.0 * (F.col("curr_year_spend") - F.col("prev_year_spend")) / F.col("prev_year_spend"), 2)
).select("year", "product_id", "curr_year_spend", "prev_year_spend", "yoy_rate")
```

#### **Senior Data Engineer Perspective**
If a product has zero sales in 2021 but sales in 2020 and 2022, `LAG()` will falsely compare 2022 to 2020. A Senior DE populates the base DataFrame with a "Date Dimension" spine to enforce `0` values for missing years before applying window functions.

---

### **Hard Lesson 3: Amazon - "Maximize Prime Item Inventory"**

#### **The Problem**
Fill a 500,000 sq ft warehouse with as many complete batches of Prime items as possible, then fill the remainder with batches of Non-Prime items.

#### **The Solution (PostgreSQL)**
```sql
WITH summary AS (
  SELECT item_type, SUM(square_footage) AS total_sqft, COUNT(item_id) AS item_count
  FROM inventory
  GROUP BY item_type
),
prime_calc AS (
  SELECT item_type, total_sqft, item_count,
    FLOOR(500000 / total_sqft) * item_count AS prime_items,
    500000 % total_sqft AS remaining_space
  FROM summary
  WHERE item_type = 'prime_eligible'
)
SELECT 'prime_eligible' AS item_type, prime_items AS item_count FROM prime_calc
UNION ALL
SELECT 'not_prime' AS item_type, FLOOR(p.remaining_space / s.total_sqft) * s.item_count AS item_count
FROM summary s CROSS JOIN prime_calc p
WHERE s.item_type = 'not_prime';
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

summary = inventory.groupBy("item_type").agg(
    F.sum("square_footage").alias("total_sqft"),
    F.count("item_id").alias("item_count")
)

# Extract integer variables for the Prime calculations to avoid complex cross-joins
prime_row = summary.filter(F.col("item_type") == "prime_eligible").collect()[0]
prime_sqft = prime_row["total_sqft"]
prime_count = prime_row["item_count"]

prime_batches = 500000 // prime_sqft
prime_items = prime_batches * prime_count
remaining_space = 500000 % prime_sqft

# Calculate Non-Prime using the remaining space variable
result_prime = spark.createDataFrame([("prime_eligible", prime_items)], ["item_type", "item_count"])

result_non_prime = summary.filter(F.col("item_type") == "not_prime").withColumn(
    "item_count", F.floor(F.lit(remaining_space) / F.col("total_sqft")) * F.col("item_count")
).select("item_type", "item_count")

result = result_prime.unionAll(result_non_prime)
```

#### **Senior Data Engineer Perspective**
This is an algorithmic bin-packing problem. In PySpark, extracting scalar values via `.collect()[0]` and passing them as literal variables to the next DataFrame is infinitely more efficient than forcing a `crossJoin()` on an aggregator node.

---

### **Hard Lesson 4: Google - "Median Google Search Frequency"**

#### **The Problem**
Find the median searches given a compressed table (search count and number of users who made that count).

#### **The Solution (PostgreSQL)**
```sql
WITH expanded_searches AS (
  SELECT searches FROM search_frequency, GENERATE_SERIES(1, num_users)
)
SELECT ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY searches)::DECIMAL, 1) AS median
FROM expanded_searches;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

# Unnest the compressed data using array_repeat and explode
expanded = search_frequency.withColumn(
    "searches_array", F.expr("array_repeat(searches, num_users)")
).withColumn(
    "exploded_searches", F.explode("searches_array")
)

# Use approxQuantile to find the median without sorting the entire dataset globally
median_val = expanded.approxQuantile("exploded_searches", [0.5], 0.01)[0]

result = spark.createDataFrame([(round(median_val, 1),)], ["median"])
```

#### **Senior Data Engineer Perspective**
`GENERATE_SERIES` (or Spark's `explode(array_repeat())`) inflates data massively. A Senior DE relies on Spark's `approxQuantile` to calculate medians. The `0.01` parameter represents the relative error margin—this allows Spark to compute the median mathematically across distributed workers without bottlenecking memory.

---

### **Hard Lesson 5: Facebook - "Advertiser Status"**

#### **The Problem**
Update the status of Facebook advertisers (New, Existing, Churn, Resurrect) based on their activity today vs. yesterday.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  COALESCE(a.user_id, p.user_id) AS user_id,
  CASE 
    WHEN p.spend IS NULL THEN 'CHURN'
    WHEN a.status = 'CHURN' AND p.spend IS NOT NULL THEN 'RESURRECT'
    WHEN a.status IS NULL AND p.spend IS NOT NULL THEN 'NEW'
    ELSE 'EXISTING'
  END AS new_status
FROM advertiser a
FULL OUTER JOIN daily_pay p ON a.user_id = p.user_id
ORDER BY user_id;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

joined = advertiser.alias("a").join(
    daily_pay.alias("p"), 
    F.col("a.user_id") == F.col("p.user_id"), 
    "full"
)

result = joined.withColumn(
    "final_user_id", F.coalesce(F.col("a.user_id"), F.col("p.user_id"))
).withColumn(
    "new_status",
    F.when(F.col("p.spend").isNull(), "CHURN")
     .when((F.col("a.status") == "CHURN") & F.col("p.spend").isNotNull(), "RESURRECT")
     .when(F.col("a.status").isNull() & F.col("p.spend").isNotNull(), "NEW")
     .otherwise("EXISTING")
).select(F.col("final_user_id").alias("user_id"), "new_status").orderBy("user_id")
```

#### **Senior Data Engineer Perspective**
In architectures like Delta Lake, maintaining state is done via `MERGE INTO` rather than full outer joins. A Senior DE ensures daily snapshots are partitioned and historical states preserved (SCD Type 2) for Data Science ML models.

---

### **Hard Lesson 6: Stripe - "Repeated Payments"**

#### **The Problem**
Find payments made by the same merchant to the same customer for the exact same amount within 10 minutes of the previous transaction.

#### **The Solution (PostgreSQL)**
```sql
WITH lagged_transactions AS (
  SELECT merchant_id, credit_card_id, amount, transaction_timestamp,
    LAG(transaction_timestamp) OVER (
      PARTITION BY merchant_id, credit_card_id, amount ORDER BY transaction_timestamp
    ) AS prev_timestamp
  FROM transactions
)
SELECT COUNT(merchant_id) AS payment_count
FROM lagged_transactions
WHERE prev_timestamp IS NOT NULL
  AND EXTRACT(EPOCH FROM (transaction_timestamp - prev_timestamp))/60 <= 10;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("merchant_id", "credit_card_id", "amount").orderBy("transaction_timestamp")

lagged = transactions.withColumn(
    "prev_timestamp", F.lag("transaction_timestamp").over(window_spec)
)

# Convert timestamps to unix seconds for mathematical difference calculation
result = lagged.filter(
    F.col("prev_timestamp").isNotNull()
).withColumn(
    "time_diff_mins", 
    (F.unix_timestamp("transaction_timestamp") - F.unix_timestamp("prev_timestamp")) / 60
).filter(
    F.col("time_diff_mins") <= 10
).select(F.count("merchant_id").alias("payment_count"))
```

#### **Senior Data Engineer Perspective**
Batch SQL is useless for stopping live duplicate charges. A Senior DE builds this in a streaming pipeline (Flink/Spark Structured Streaming) holding transactions in a 10-minute stateful window to actively drop duplicates in real-time.

---

### **Hard Lesson 7: McKinsey - "3-Topping Pizzas"**

#### **The Problem**
Find all combinations of 3 different pizza toppings sorted alphabetically.

#### **The Solution (PostgreSQL)**
```sql
SELECT CONCAT(t1.topping_name, ',', t2.topping_name, ',', t3.topping_name) AS pizza,
       t1.topping_price + t2.topping_price + t3.topping_price AS total_cost
FROM pizza_toppings t1
JOIN pizza_toppings t2 ON t1.topping_name < t2.topping_name
JOIN pizza_toppings t3 ON t2.topping_name < t3.topping_name
ORDER BY total_cost DESC, pizza ASC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

t1 = pizza_toppings.alias("t1")
t2 = pizza_toppings.alias("t2")
t3 = pizza_toppings.alias("t3")

# Using inner joins with inequality conditions creates a filtered Cartesian product
result = t1.join(t2, F.col("t1.topping_name") < F.col("t2.topping_name"), "inner") \
           .join(t3, F.col("t2.topping_name") < F.col("t3.topping_name"), "inner") \
           .select(
               F.concat_ws(",", F.col("t1.topping_name"), F.col("t2.topping_name"), F.col("t3.topping_name")).alias("pizza"),
               (F.col("t1.topping_price") + F.col("t2.topping_price") + F.col("t3.topping_price")).alias("total_cost")
           ).orderBy(F.col("total_cost").desc(), F.col("pizza").asc())
```

#### **Senior Data Engineer Perspective**
Inequalities in `JOIN` conditions essentially force a Cross Join. Applying this to large tables (e.g., finding "3-person friend groups" among millions of users) crashes Spark clusters with OOM errors instantly. Always evaluate dataset size before using combinatorics.

---

### **Hard Lesson 8: Intuit - "Consecutive Filing Years"**

#### **The Problem**
Find users who filed taxes for 3 or more consecutive years.

#### **The Solution (PostgreSQL)**
```sql
WITH distinct_filings AS (
  SELECT DISTINCT user_id, filing_year FROM tax_filings
),
island_groups AS (
  SELECT user_id, filing_year,
    filing_year - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY filing_year) AS streak_group
  FROM distinct_filings
)
SELECT user_id, COUNT(filing_year) AS consecutive_years
FROM island_groups
GROUP BY user_id, streak_group
HAVING COUNT(filing_year) >= 3;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

distinct_filings = tax_filings.select("user_id", "filing_year").distinct()

window_spec = Window.partitionBy("user_id").orderBy("filing_year")

island_groups = distinct_filings.withColumn(
    "streak_group", 
    F.col("filing_year") - F.row_number().over(window_spec)
)

result = island_groups.groupBy("user_id", "streak_group").agg(
    F.count("filing_year").alias("consecutive_years")
).filter(F.col("consecutive_years") >= 3).select("user_id", "consecutive_years")
```

#### **Senior Data Engineer Perspective**
Always enforce uniqueness first (`.distinct()`). If source data has duplicate amendments for the same year, the `ROW_NUMBER` subtraction trick breaks completely. 

---

### **Hard Lesson 9: Amazon - "Server Utilization Time"**

#### **The Problem**
Calculate total uptime in full days across all servers using a log of start and stop events.

#### **The Solution (PostgreSQL)**
```sql
WITH session_pairs AS (
  SELECT server_id, session_status, status_time AS start_time,
    LEAD(status_time) OVER (PARTITION BY server_id ORDER BY status_time) AS stop_time
  FROM server_utilization
)
SELECT EXTRACT(DAY FROM SUM(stop_time - start_time)) AS total_uptime_days
FROM session_pairs
WHERE session_status = 'start';
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("server_id").orderBy("status_time")

session_pairs = server_utilization.withColumn(
    "stop_time", F.lead("status_time").over(window_spec)
).filter(F.col("session_status") == "start")

# Convert to unix_timestamp to calculate difference in seconds, then divide to get days
result = session_pairs.select(
    F.floor(
        F.sum(F.unix_timestamp("stop_time") - F.unix_timestamp("status_time")) / 86400
    ).alias("total_uptime_days")
)
```

#### **Senior Data Engineer Perspective**
If a server crashes and never logs a "stop", `LEAD()` returns `NULL`, breaking the `SUM()`. Wrap the `LEAD()` function in a `COALESCE(..., F.current_timestamp())` to safely close out active sessions.

---

### **Hard Lesson 10: Snowflake - "Marketing Touch Streak"**

#### **The Problem**
Find contacts with a marketing touch for 3+ consecutive weeks who also had a `'trial_request'`.

#### **The Solution (PostgreSQL)**
```sql
WITH weekly_touches AS (
  SELECT DISTINCT contact_id, DATE_TRUNC('week', event_date) AS touch_week
  FROM marketing_touches
),
streak_calc AS (
  SELECT contact_id, touch_week,
    LAG(touch_week) OVER (PARTITION BY contact_id ORDER BY touch_week) AS prev_week,
    LEAD(touch_week) OVER (PARTITION BY contact_id ORDER BY touch_week) AS next_week
  FROM weekly_touches
)
SELECT DISTINCT c.email
FROM streak_calc s
JOIN crm_contacts c ON s.contact_id = c.contact_id
JOIN marketing_touches m ON s.contact_id = m.contact_id
WHERE s.touch_week - INTERVAL '1 week' = s.prev_week 
  AND s.touch_week + INTERVAL '1 week' = s.next_week
  AND m.event_type = 'trial_request';
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

weekly_touches = marketing_touches.withColumn(
    "touch_week", F.date_trunc("week", "event_date")
).select("contact_id", "touch_week").distinct()

window_spec = Window.partitionBy("contact_id").orderBy("touch_week")

streak_calc = weekly_touches.withColumn("prev_week", F.lag("touch_week").over(window_spec)) \
                            .withColumn("next_week", F.lead("touch_week").over(window_spec))

# Use date_sub and date_add to verify strict 7 day spacing
valid_streaks = streak_calc.filter(
    (F.date_sub("touch_week", 7) == F.col("prev_week")) & 
    (F.date_add("touch_week", 7) == F.col("next_week"))
)

trials = marketing_touches.filter(F.col("event_type") == "trial_request").select("contact_id").distinct()

result = valid_streaks.join(trials, "contact_id", "inner") \
                      .join(crm_contacts, "contact_id", "inner") \
                      .select("email").distinct()
```

#### **Senior Data Engineer Perspective**
The `DISTINCT` in the CTE is critical. Multiple touches in the *same* week break the `LAG()` logic. Normalize time-series data to the required grain (weekly) before running sequence analytics.

---

### **Hard Lesson 11: UnitedHealth - "Patient Support Analysis (Part 3)"**

#### **The Problem**
Calculate the exact time difference between consecutive calls made by the same policyholder.

#### **The Solution (PostgreSQL)**
```sql
SELECT policy_holder_id, call_date AS current_call, 
  LAG(call_date) OVER (PARTITION BY policy_holder_id ORDER BY call_date) AS previous_call, 
  call_date - LAG(call_date) OVER (PARTITION BY policy_holder_id ORDER BY call_date) AS time_difference 
FROM callers;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("policy_holder_id").orderBy("call_date")

result = callers.withColumn(
    "previous_call", F.lag("call_date").over(window_spec)
).withColumn(
    "time_difference_seconds", 
    F.unix_timestamp("call_date") - F.unix_timestamp("previous_call")
).select(
    "policy_holder_id", F.col("call_date").alias("current_call"), "previous_call", "time_difference_seconds"
)
```

#### **Senior Data Engineer Perspective**
PostgreSQL easily returns `INTERVAL` types, but BI tools struggle to chart them. PySpark forces you to convert the timestamps to Unix epochs (seconds), resulting in a standardized integer that is universally supported by dashboards.

---

### **Hard Lesson 12: UnitedHealth - "Patient Support Analysis (Part 4)"**

#### **The Problem**
Find the number of policyholders who called within exactly 7 days of their previous call.

#### **The Solution (PostgreSQL)**
```sql
WITH call_history AS (
  SELECT policy_holder_id,
    call_date - LAG(call_date) OVER (PARTITION BY policy_holder_id ORDER BY call_date) AS time_between_calls
  FROM callers
)
SELECT COUNT(DISTINCT policy_holder_id) AS frequent_callers
FROM call_history
WHERE EXTRACT(DAY FROM time_between_calls) <= 7;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`, `pyspark.sql.window.Window`
```python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("policy_holder_id").orderBy("call_date")

call_history = callers.withColumn(
    "time_between_calls_days", 
    F.datediff("call_date", F.lag("call_date").over(window_spec))
)

result = call_history.filter(
    F.col("time_between_calls_days") <= 7
).select(F.countDistinct("policy_holder_id").alias("frequent_callers"))
```

#### **Senior Data Engineer Perspective**
Windowing by ID is efficient unless you have "super-callers" (e.g., an automated system or corporate hotline). Monitor DAG execution for straggler tasks in the Spark UI, which indicates data skew on a specific partition key.

---

### **Hard Lesson 13: Facebook - "Reactivated Users"**

#### **The Problem**
Count users who did not log in the previous month but logged in the current month.

#### **The Solution (PostgreSQL)**
```sql
SELECT EXTRACT(MONTH FROM curr_month.login_date) AS mth, COUNT(DISTINCT curr_month.user_id) AS reactivated_users 
FROM user_logins AS curr_month 
WHERE NOT EXISTS ( 
  SELECT 1 FROM user_logins AS last_month 
  WHERE curr_month.user_id = last_month.user_id 
    AND EXTRACT(MONTH FROM last_month.login_date) = EXTRACT(MONTH FROM curr_month.login_date - INTERVAL '1 month') 
)
GROUP BY EXTRACT(MONTH FROM curr_month.login_date)
ORDER BY mth;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

# Truncate to month to avoid day-math edge cases (like Feb 28 vs Jan 31)
logins = user_logins.withColumn("login_month", F.date_trunc("month", "login_date"))

curr_month = logins.alias("curr")
# Calculate the exact prior month for joining purposes
prev_month_logins = logins.withColumn("target_next_month", F.expr("add_months(login_month, 1)")).alias("prev")

# Left Anti Join removes users who had a login in the target previous month
reactivated = curr_month.join(
    prev_month_logins,
    (F.col("curr.user_id") == F.col("prev.user_id")) & 
    (F.col("curr.login_month") == F.col("prev.target_next_month")),
    "left_anti"
)

result = reactivated.withColumn("mth", F.month("login_date")).groupBy("mth").agg(
    F.countDistinct("user_id").alias("reactivated_users")
).orderBy("mth")
```

#### **Senior Data Engineer Perspective**
Subtracting `- INTERVAL '1 month'` works in Postgres but causes edge-case errors in Spark on dates like March 31. It is vastly safer to use `date_trunc` to align everything to the 1st of the month before executing interval math or joins.

---

### **Hard Lesson 14: Google - "Senior Managers"**

#### **The Problem**
Find "senior managers" (managers who manage other managers) and count their direct manager reports.

#### **The Solution (PostgreSQL)**
```sql
SELECT senior_managers.manager_name, COUNT(DISTINCT managers.emp_id) AS direct_reportees 
FROM employees
JOIN employees AS managers ON employees.manager_id = managers.emp_id 
JOIN employees AS senior_managers ON managers.manager_id = senior_managers.emp_id 
GROUP BY senior_managers.manager_name 
ORDER BY direct_reportees DESC;
```

#### **The Solution (PySpark)**
**Modules Used:** `pyspark.sql.functions as F`
```python
import pyspark.sql.functions as F

base_emp = employees.alias("base")
managers = employees.alias("mgr")
senior_mgr = employees.alias("snr_mgr")

joined = base_emp.join(
    managers, F.col("base.manager_id") == F.col("mgr.emp_id"), "inner"
).join(
    senior_mgr, F.col("mgr.manager_id") == F.col("snr_mgr.emp_id"), "inner"
)

result = joined.groupBy(F.col("snr_mgr.manager_name")).agg(
    F.countDistinct("mgr.emp_id").alias("direct_reportees")
).orderBy(F.col("direct_reportees").desc())
```

#### **Senior Data Engineer Perspective**
Recursive structures are ugly in distributed SQL because they require explicit self-joins for every depth level. At FAANG scale, a Senior DE architects HR hierarchies using Graph Databases (like Amazon Neptune) or Spark GraphX to compute network dependencies without hardcoded joins.